In [1]:
# ===============================
# 0. Dependencies & Setup
# ===============================
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install -q torch_geometric matplotlib tqdm scikit-learn

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

import matplotlib.pyplot as plt
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/Honors/'
TEXT_FEAT = 'text_features_imp.npy'
AUDIO_FEAT = 'audio_features_imp.npy'
CONTEXT_FEAT = 'context_features_imp.npy'
LABELS_FEAT = 'labels_imp.npy'

BATCH_SIZE = 16
EPOCHS = 150
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3  # Increased to penalize complex models and reduce overfitting
PATIENCE = 20

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.4 MB/s eta 0:00:00
Using device: cuda


In [2]:
# ===============================
# 1. Load & Split Features
# ===============================
from google.colab import drive
drive.mount('/content/drive')

print("Loading Text, Audio, and Context Features...")
t_features = np.load(os.path.join(BASE_PATH, TEXT_FEAT))
a_features = np.load(os.path.join(BASE_PATH, AUDIO_FEAT))
c_features = np.load(os.path.join(BASE_PATH, CONTEXT_FEAT))
labels     = np.load(os.path.join(BASE_PATH, LABELS_FEAT))

print(f"Loaded {len(labels)} samples. Dims: T={t_features.shape[1]}, A={a_features.shape[1]}, C={c_features.shape[1]}")

data_list = []
for i in range(len(labels)):
    t = torch.tensor(t_features[i], dtype=torch.float).unsqueeze(0)
    a = torch.tensor(a_features[i], dtype=torch.float).unsqueeze(0)
    c = torch.tensor(c_features[i], dtype=torch.float).unsqueeze(0)
    y = torch.tensor([labels[i]], dtype=torch.float)

    edge_index = torch.tensor([
        [0,1],[1,0], [0,2],[2,0], [1,2],[2,1],
        [0,0],[1,1],[2,2]
    ], dtype=torch.long).t().contiguous()

    data = Data(edge_index=edge_index, y=y)
    data.text, data.audio, data.context, data.num_nodes = t, a, c, 3
    data_list.append(data)

# 70/15/15 Split
train_val, test = train_test_split(data_list, test_size=0.15, random_state=42, stratify=labels)
labels_train_val = np.array([d.y.item() for d in train_val])
train, val = train_test_split(train_val, test_size=0.1765, random_state=42, stratify=labels_train_val)

train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test,  batch_size=BATCH_SIZE, shuffle=False)

Mounted at /content/drive
Loading Text, Audio, and Context Features...
Loaded 690 samples. Dims: T=768, A=768, C=2816


In [3]:
# ===============================
# 2. TACGATv2 with Attention Fusion
# ===============================
class TACGATv2_Improved(nn.Module):
    def __init__(self, t_dim, a_dim, c_dim, hidden_dim=128, heads=4, dropout=0.5):
        super().__init__()
        # Projections to common hidden space
        self.p_t = nn.Sequential(nn.Linear(t_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.p_a = nn.Sequential(nn.Linear(a_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.p_c = nn.Sequential(nn.Linear(c_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())

        self.gat1 = GATv2Conv(hidden_dim, hidden_dim, heads=heads, concat=True, dropout=dropout)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout)
        self.gat_norm = nn.LayerNorm(hidden_dim)

        # Learnable vector to compute importance scores for each modality node
        self.fusion_query = nn.Parameter(torch.randn(1, hidden_dim))

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, data):
        x_t, x_a, x_c = self.p_t(data.text), self.p_a(data.audio), self.p_c(data.context)
        batch_size = x_t.size(0)

        # Interleave: [T, A, C, T, A, C...]
        x = torch.zeros(batch_size * 3, x_t.size(1), device=DEVICE)
        x[0::3], x[1::3], x[2::3] = x_t, x_a, x_c

        base_edges = data.edge_index[:, :9]
        edge_index = torch.cat([base_edges + i * 3 for i in range(batch_size)], dim=1).to(DEVICE)

        out = F.gelu(self.gat1(x, edge_index))
        out = self.gat_norm(self.gat2(out, edge_index))

        # Reshape to [Batch, 3 Nodes, Hidden]
        h = out.view(batch_size, 3, -1)

        # Attention Weighting
        attn_scores = torch.matmul(h, self.fusion_query.t()).squeeze(-1) # [B, 3]
        attn_weights = F.softmax(attn_scores, dim=1).unsqueeze(-1)       # [B, 3, 1]

        fused = torch.sum(h * attn_weights, dim=1) # [B, Hidden]
        return self.classifier(fused).squeeze(-1)

In [4]:
# ===============================
# 3. Training & Evaluation Logic
# ===============================
def train_one_epoch(loader, model, optimizer, criterion):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for data in loader:
        data = data.to(DEVICE)
        optimizer.zero_grad()
        logits = model(data)
        loss = criterion(logits, data.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.y.size(0)
        all_preds.extend((torch.sigmoid(logits) >= 0.5).cpu().numpy())
        all_labels.extend(data.y.view(-1).cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro')

@torch.no_grad()
def evaluate(loader, model):
    model.eval()
    probs, preds, labels = [], [], []
    for data in loader:
        data = data.to(DEVICE)
        logits = model(data)
        p = torch.sigmoid(logits).cpu().numpy()
        probs.extend(p); preds.extend((p >= 0.5).astype(int)); labels.extend(data.y.view(-1).cpu().numpy())
    return accuracy_score(labels, preds), f1_score(labels, preds, average='macro'), precision_score(labels, preds, average='macro', zero_division=0), recall_score(labels, preds, average='macro', zero_division=0), roc_auc_score(labels, probs)

In [5]:
# ===============================
# 4. Run Training
# ===============================
model = TACGATv2_Improved(t_features.shape[1], a_features.shape[1], c_features.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
criterion = nn.BCEWithLogitsLoss()

best_val_f1, patience_counter = 0.0, 0
for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc, t_f1 = train_one_epoch(train_loader, model, optimizer, criterion)
    v_acc, v_f1, v_prec, v_rec, v_auc = evaluate(val_loader, model)
    scheduler.step(v_f1)

    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        torch.save(model.state_dict(), "best_sarcasm_model.pth")
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or patience_counter == 0:
        print(f"Epoch {epoch:03d} | Val F1: {v_f1:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

model.load_state_dict(torch.load("best_sarcasm_model.pth"))
test_acc, test_f1, _, _, test_auc = evaluate(test_loader, model)
print(f"\nFINAL TEST RESULTS | Accuracy: {test_acc:.4f} | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")

Epoch 001 | Val F1: 0.3333 | LR: 1.00e-04
Epoch 002 | Val F1: 0.4826 | LR: 1.00e-04
Epoch 005 | Val F1: 0.5669 | LR: 1.00e-04
Epoch 009 | Val F1: 0.6224 | LR: 1.00e-04
Epoch 010 | Val F1: 0.5117 | LR: 1.00e-04
Epoch 013 | Val F1: 0.6297 | LR: 1.00e-04
Epoch 018 | Val F1: 0.6581 | LR: 1.00e-04
Epoch 020 | Val F1: 0.6291 | LR: 1.00e-04
Epoch 027 | Val F1: 0.6609 | LR: 5.00e-05
Epoch 030 | Val F1: 0.6312 | LR: 5.00e-05
Epoch 032 | Val F1: 0.6627 | LR: 5.00e-05
Epoch 034 | Val F1: 0.6820 | LR: 5.00e-05
Epoch 036 | Val F1: 0.6824 | LR: 5.00e-05
Epoch 038 | Val F1: 0.6894 | LR: 5.00e-05
Epoch 040 | Val F1: 0.6432 | LR: 5.00e-05
Epoch 042 | Val F1: 0.6997 | LR: 5.00e-05
Epoch 046 | Val F1: 0.7006 | LR: 5.00e-05
Epoch 050 | Val F1: 0.6711 | LR: 5.00e-05
Epoch 060 | Val F1: 0.6986 | LR: 1.25e-05
Early stopping triggered.

FINAL TEST RESULTS | Accuracy: 0.7404 | F1: 0.7384 | AUC: 0.7825
